# Liu2024 Source `.mat` + S-JEPA-style preprocessing + CSP/FBCSP

This notebook intentionally avoids MOABB event/window creation for Liu2024.

It uses the original Figshare source `.mat` files, where each subject is already stored as epoched trials:

- `40 trials × 33 channels × 4000 samples`
- `500 Hz`
- labels: left/right hand MI

Then it applies the same **style** of preprocessing used in your S-JEPA notebook:

1. select EEG channels only
2. resample to 128 Hz
3. bandpass 0.5–40 Hz
4. average reference
5. crop a fixed 537-sample MI window
6. run one CSP+LDA setup and one FBCSP+LDA setup

The broken `ds.windows.get_data()` path is not used anywhere. Everything is array-based after loading the source `.mat` trials.

In [1]:
import os
import re
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.io import loadmat
from scipy.signal import butter, sosfiltfilt

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

try:
    import mne
    from mne.decoding import CSP
except Exception as exc:
    raise RuntimeError('This notebook needs MNE installed: pip install mne') from exc

print('Runtime Environment:')
print(f'  Python: {os.sys.version}')
print(f'  MNE:    {mne.__version__}')
print(f'  Workdir: {Path.cwd()}')

Runtime Environment:
  Python: 3.11.15 (main, Apr  9 2026, 01:18:52) [Clang 21.0.0 (clang-2100.0.123.102)]
  MNE:    1.11.0
  Workdir: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src


## 1. Paths and run settings

The notebook searches common project locations for `liu2024_figshare/sourcedata`. If your source files are somewhere else, set `MANUAL_SOURCE_EXTRACT_DIR` below.

In [2]:
# Optional manual override. Leave as None to auto-detect.
MANUAL_SOURCE_EXTRACT_DIR = None

WORKING_DIR = Path.cwd()
ARTIFACT_DIR = WORKING_DIR / 'artifacts' / 'liu2024-source-mat-sjepa-preprocess-csp-fbcsp' / datetime.now().strftime('%Y%m%d_%H%M%S')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / 'run.log'

# S-JEPA-style preprocessing constants.
SOURCE_SFREQ = 500
SFREQ = 128
BANDPASS_LOW = 0.5
BANDPASS_HIGH = 40.0

# Pierre / S-JEPA-compatible downstream length.
TARGET_WINDOW_SAMPLES = 537
TARGET_WINDOW_DURATION_S = TARGET_WINDOW_SAMPLES / SFREQ

# Liu source trial timing: 8 seconds total at 500 Hz.
# For S-JEPA-style comparison we crop the MI interval starting at 2s after resampling.
MI_WINDOW_START_S = 2.0
MI_WINDOW_START_SAMPLE = int(round(MI_WINDOW_START_S * SFREQ))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + TARGET_WINDOW_SAMPLES

CV_FOLDS = 5
RANDOM_STATE = 2026

# CSP/FBCSP setup from your S-JEPA-style baseline family.
CSP_BAND = (8.0, 30.0)
FBCSP_BANDS = [(4, 8), (8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 32), (32, 36), (36, 40)]
N_CSP_COMPONENTS = 4

# Source channel convention from Liu paper / observed source files:
# rawdata shape: trials x 33 channels x 4000 samples
# 0..29 = EEG-like channels, index 17 = CPz reference channel, 30..31 = EOG, 32 = marker.
# MOABB exposes 29 EEG channels, so we drop CPz reference, EOG, and marker.
EEG_CHANNEL_INDICES_29 = [i for i in range(30) if i != 17]
EEG_CHANNEL_NAMES_29 = [
    'Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'FCz', 'FC3', 'FC4',
    'FT7', 'FT8', 'Cz', 'C3', 'C4', 'T7', 'T8',
    # skip CPz original reference at source index 17
    'CP3', 'CP4', 'TP7', 'TP8', 'Pz', 'P3', 'P4', 'P7', 'P8', 'Oz', 'O1', 'O2'
]

print(f'Artifacts: {ARTIFACT_DIR}')
print(f'Target window: {TARGET_WINDOW_SAMPLES} samples = {TARGET_WINDOW_DURATION_S:.6f}s')
print(f'Crop: {MI_WINDOW_START_SAMPLE}:{MI_WINDOW_STOP_SAMPLE} at {SFREQ} Hz')

Artifacts: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-preprocess-csp-fbcsp/20260518_192347
Target window: 537 samples = 4.195312s
Crop: 256:793 at 128 Hz


In [3]:
_log_handle = open(LOG_PATH, 'w', buffering=1)

def log(msg=''):
    text = str(msg)
    print(text)
    _log_handle.write(text + '\n')

log('Liu2024 source-mat S-JEPA-style preprocessing CSP/FBCSP run')
log(f'Artifacts: {ARTIFACT_DIR}')
log(f'Target window samples: {TARGET_WINDOW_SAMPLES}')
log(f'Effective target duration: {TARGET_WINDOW_DURATION_S:.6f}s')

Liu2024 source-mat S-JEPA-style preprocessing CSP/FBCSP run
Artifacts: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-preprocess-csp-fbcsp/20260518_192347
Target window samples: 537
Effective target duration: 4.195312s


## 2. Locate Liu2024 source `.mat` files

This should point to the extracted Figshare `sourcedata` folder, not MOABB cache files.

In [4]:
def find_source_mat_files(root: Path):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(root.rglob('*.mat'))


def candidate_source_dirs():
    candidates = []
    if MANUAL_SOURCE_EXTRACT_DIR is not None:
        candidates.append(Path(MANUAL_SOURCE_EXTRACT_DIR))

    # Common layouts depending on where the notebook is run from.
    for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        candidates.extend([
            base / 'liu2024_figshare' / 'sourcedata',
            base / 'liu2024_figshare' / 'sourcedata' / 'sourcedata',
            base / 'src' / 'liu2024_figshare' / 'sourcedata',
            base / 'src' / 'liu2024_figshare' / 'sourcedata' / 'sourcedata',
        ])
    seen = set()
    uniq = []
    for c in candidates:
        key = str(c.resolve()) if c.exists() else str(c)
        if key not in seen:
            uniq.append(c)
            seen.add(key)
    return uniq


MAT_FILES = []
SOURCE_EXTRACT_DIR = None
for cand in candidate_source_dirs():
    files = find_source_mat_files(cand)
    if files:
        MAT_FILES = files
        SOURCE_EXTRACT_DIR = cand
        break

if not MAT_FILES:
    log('Could not auto-find Liu2024 source .mat files.')
    log('Checked these candidate directories:')
    for c in candidate_source_dirs():
        log(f'  {c}')
    raise FileNotFoundError('Set MANUAL_SOURCE_EXTRACT_DIR to your extracted Figshare sourcedata directory.')

log(f'Source extract dir: {SOURCE_EXTRACT_DIR}')
log(f'Found {len(MAT_FILES)} .mat files')
log('First 5 files:')
for p in MAT_FILES[:5]:
    log(f'  {p}')

Source extract dir: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata
Found 50 .mat files
First 5 files:
  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-01/sub-01_task-motor-imagery_eeg.mat
  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-02/sub-02_task-motor-imagery_eeg.mat
  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-03/sub-03_task-motor-imagery_eeg.mat
  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-04/sub-04_task-motor-imagery_eeg.mat
  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-05/sub-05_task-motor-imagery_eeg.mat


## 3. Robust source `.mat` loader

The Figshare files may store arrays under an `eeg` MATLAB struct, not top-level `rawdata` / `labels`. This loader recursively searches nested structs and verifies that it found a 3D data array plus a label vector.

In [5]:
def subject_id_from_path(path: Path):
    s = str(path)
    m = re.search(r'sub[-_ ]?(\d{1,2})', s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r'\d+', Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f'Could not infer subject id from path: {path}')


def _is_mat_struct(x):
    return hasattr(x, '_fieldnames')


def _walk_mat_object(obj, prefix=''):
    # Yield (name, value) recursively from scipy-loaded MATLAB structs.
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k.startswith('__'):
                continue
            name = f'{prefix}.{k}' if prefix else k
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f'{prefix}.{k}' if prefix else k
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                name = f'{prefix}{idx}'
                yield from _walk_mat_object(item, name)


def mat_structure_preview(path: Path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({'name': name, 'type': 'ndarray', 'shape': str(value.shape), 'dtype': str(value.dtype)})
        else:
            rows.append({'name': name, 'type': type(value).__name__, 'shape': '', 'dtype': ''})
    return pd.DataFrame(rows).head(max_rows)


def load_subject_mat(path: Path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates = []
    label_candidates = []
    for name, arr in arrays:
        lname = name.lower()
        if arr.ndim == 3:
            score = 0
            if 'rawdata' in lname or 'data' in lname:
                score += 10
            if arr.shape[0] == 40 or arr.shape[1] == 40:
                score += 2
            if 3000 <= max(arr.shape) <= 5000:
                score += 2
            raw_candidates.append((score, name, arr))
        elif arr.ndim in (1, 2):
            flat = arr.ravel()
            unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
            score = 0
            if 'label' in lname or 'class' in lname or 'y' == lname.split('.')[-1]:
                score += 10
            if flat.size in (40, 39):
                score += 3
            if unique and unique.issubset({'1', '2', '0'}):
                score += 2
            label_candidates.append((score, name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        log(f'MAT structure preview for failure at {path}:')
        log(preview.to_string(index=False))
        raise KeyError(f'Could not locate 3D raw data and labels in {path}')

    raw_candidates = sorted(raw_candidates, key=lambda x: x[0], reverse=True)
    label_candidates = sorted(label_candidates, key=lambda x: x[0], reverse=True)
    raw_name, rawdata = raw_candidates[0][1], np.asarray(raw_candidates[0][2])
    label_name, labels = label_candidates[0][1], np.asarray(label_candidates[0][2]).astype(int).ravel()

    if rawdata.ndim != 3:
        raise ValueError(f'rawdata must be 3D, got {rawdata.shape}')
    if rawdata.shape[0] != labels.size:
        trial_axis = None
        for ax, size in enumerate(rawdata.shape):
            if size == labels.size:
                trial_axis = ax
                break
        if trial_axis is None:
            raise ValueError(f'Cannot align labels {labels.shape} with rawdata {rawdata.shape} in {path}')
        rawdata = np.moveaxis(rawdata, trial_axis, 0)

    if rawdata.shape[-1] < rawdata.shape[1]:
        rawdata = np.swapaxes(rawdata, 1, 2)

    return rawdata.astype(np.float64), labels.astype(int), raw_name, label_name


preview = mat_structure_preview(MAT_FILES[0])
preview_path = ARTIFACT_DIR / 'mat_structure_subject_01.csv'
preview.to_csv(preview_path, index=False)
log(f'Wrote MAT structure preview: {preview_path}')
display(preview.head(20))

Wrote MAT structure preview: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-preprocess-csp-fbcsp/20260518_192347/mat_structure_subject_01.csv


,name,type,shape,dtype
0,eeg,mat_struct,,
1,eeg.rawdata,ndarray,"(40, 33, 4000)",float64
2,eeg.label,ndarray,"(40,)",uint8


## 4. Load and verify all subjects

Expected: 50 subjects, 40 trials per subject, balanced 20/20 labels.

In [6]:
subjects = []
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    X_raw, y_raw, raw_field, label_field = load_subject_mat(p)
    subjects.append({
        'subject_id': sid,
        'path': str(p),
        'rawdata_shape': tuple(X_raw.shape),
        'labels_shape': tuple(y_raw.shape),
        'label_counts_raw': np.bincount(y_raw.astype(int), minlength=3).tolist(),
        'raw_field': raw_field,
        'label_field': label_field,
    })

subjects_df = pd.DataFrame(subjects).sort_values('subject_id')
subjects_df.to_csv(ARTIFACT_DIR / 'source_mat_summary.csv', index=False)
log(f'Wrote source summary: {ARTIFACT_DIR / "source_mat_summary.csv"}')
display(subjects_df.head())
log(subjects_df[['subject_id', 'rawdata_shape', 'labels_shape', 'label_counts_raw', 'raw_field', 'label_field']].head().to_string(index=False))

Wrote source summary: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-preprocess-csp-fbcsp/20260518_192347/source_mat_summary.csv


,subject_id,path,rawdata_shape,labels_shape,label_counts_raw,raw_field,label_field
0,1,/Users/vadim/Documents/School/Spring 2026/CSCE...,"(40, 33, 4000)","(40,)","[0, 20, 20]",eeg.rawdata,eeg.label
1,2,/Users/vadim/Documents/School/Spring 2026/CSCE...,"(40, 33, 4000)","(40,)","[0, 20, 20]",eeg.rawdata,eeg.label
2,3,/Users/vadim/Documents/School/Spring 2026/CSCE...,"(40, 33, 4000)","(40,)","[0, 20, 20]",eeg.rawdata,eeg.label
3,4,/Users/vadim/Documents/School/Spring 2026/CSCE...,"(40, 33, 4000)","(40,)","[0, 20, 20]",eeg.rawdata,eeg.label
4,5,/Users/vadim/Documents/School/Spring 2026/CSCE...,"(40, 33, 4000)","(40,)","[0, 20, 20]",eeg.rawdata,eeg.label


 subject_id  rawdata_shape labels_shape label_counts_raw   raw_field label_field
          1 (40, 33, 4000)        (40,)      [0, 20, 20] eeg.rawdata   eeg.label
          2 (40, 33, 4000)        (40,)      [0, 20, 20] eeg.rawdata   eeg.label
          3 (40, 33, 4000)        (40,)      [0, 20, 20] eeg.rawdata   eeg.label
          4 (40, 33, 4000)        (40,)      [0, 20, 20] eeg.rawdata   eeg.label
          5 (40, 33, 4000)        (40,)      [0, 20, 20] eeg.rawdata   eeg.label


## 5. S-JEPA-style preprocessing from source trials

This cell converts each subject into fixed windows as arrays:

```text
source .mat trials: 40 × 33 × 4000 at 500 Hz
select EEG only:    40 × 29 × 4000
resample/filter/ref:40 × 29 × 1024 at 128 Hz
crop MI window:     40 × 29 × 537
```

The preprocessing order mirrors the S-JEPA-style MOABB notebook:

```text
pick EEG → resample 128 Hz → filter 0.5–40 Hz → average reference
```

Because source `.mat` files are already epoched, this notebook applies MNE preprocessing by concatenating the 40 trials into one subject-level RawArray, then reshaping back into trials. This avoids the broken Braindecode `EEGWindowsDataset.windows` path and keeps the source-trial structure intact.

In [7]:
def preprocess_subject_sjepa_style(rawdata, labels, subject_id):
    # Return X_win, y for one subject using source .mat trial data.
    # rawdata: trials x 33 channels x 4000 samples at 500 Hz.
    # labels: 1/2 labels from Liu source files.
    if rawdata.shape[1] < 33:
        raise ValueError(f'Expected at least 33 channels, got {rawdata.shape}')
    if rawdata.shape[2] != 4000:
        log(f'WARNING subject {subject_id}: expected 4000 samples, got {rawdata.shape[2]}')

    n_trials = rawdata.shape[0]
    X_eeg = rawdata[:, EEG_CHANNEL_INDICES_29, :].astype(np.float64)

    continuous = X_eeg.transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES_29), -1)
    info = mne.create_info(
        ch_names=EEG_CHANNEL_NAMES_29,
        sfreq=SOURCE_SFREQ,
        ch_types=['eeg'] * len(EEG_CHANNEL_NAMES_29),
    )
    raw = mne.io.RawArray(continuous, info, verbose=False)

    # S-JEPA-style order from your MOABB notebook.
    raw.resample(SFREQ, verbose=False)
    raw.filter(BANDPASS_LOW, BANDPASS_HIGH, verbose=False)
    raw.set_eeg_reference('average', projection=False, verbose=False)

    data = raw.get_data()
    expected_samples_per_trial = int(round(rawdata.shape[2] * SFREQ / SOURCE_SFREQ))
    total_expected = n_trials * expected_samples_per_trial

    if data.shape[1] != total_expected:
        n_full = data.shape[1] // n_trials
        log(
            f'WARNING subject {subject_id}: resampled samples {data.shape[1]} != expected {total_expected}; '
            f'using {n_full} samples/trial.'
        )
        expected_samples_per_trial = n_full
        data = data[:, :n_trials * expected_samples_per_trial]

    X_rs = data.reshape(len(EEG_CHANNEL_INDICES_29), n_trials, expected_samples_per_trial).transpose(1, 0, 2)

    if MI_WINDOW_STOP_SAMPLE > X_rs.shape[-1]:
        raise ValueError(
            f'MI crop {MI_WINDOW_START_SAMPLE}:{MI_WINDOW_STOP_SAMPLE} exceeds resampled trial length {X_rs.shape[-1]}'
        )

    X_win = X_rs[:, :, MI_WINDOW_START_SAMPLE:MI_WINDOW_STOP_SAMPLE]
    y = labels.astype(int) - 1
    if not set(np.unique(y)).issubset({0, 1}):
        raise ValueError(f'Unexpected labels after conversion for subject {subject_id}: {np.unique(y)}')

    return X_win.astype(np.float32), y.astype(int), X_rs.shape[-1]


Xs, ys, subjects_arr = [], [], []
window_summary_rows = []

for item in subjects_df.to_dict('records'):
    sid = int(item['subject_id'])
    X_raw, y_raw, raw_field, label_field = load_subject_mat(Path(item['path']))
    X_win, y, samples_per_trial = preprocess_subject_sjepa_style(X_raw, y_raw, sid)

    Xs.append(X_win)
    ys.append(y)
    subjects_arr.extend([sid] * len(y))

    window_summary_rows.append({
        'subject_id': sid,
        'n_windows': int(len(y)),
        'class_counts': np.bincount(y, minlength=2).tolist(),
        'preprocessed_shape': tuple(X_win.shape),
        'resampled_samples_per_trial': int(samples_per_trial),
        'crop_start_sample': MI_WINDOW_START_SAMPLE,
        'crop_stop_sample': MI_WINDOW_STOP_SAMPLE,
        'target_window_samples': TARGET_WINDOW_SAMPLES,
        'effective_window_duration_s': TARGET_WINDOW_DURATION_S,
    })

X_all = np.concatenate(Xs, axis=0)
y_all = np.concatenate(ys, axis=0)
subjects_arr = np.asarray(subjects_arr)

window_summary_df = pd.DataFrame(window_summary_rows).sort_values('subject_id')
window_summary_df.to_csv(ARTIFACT_DIR / 'window_counts_by_subject.csv', index=False)

log(f'X_all shape: {X_all.shape}')
log(f'y_all counts: {np.bincount(y_all, minlength=2).tolist()}')
log(f'subjects_arr shape: {subjects_arr.shape}')
log(f'Wrote window summary: {ARTIFACT_DIR / "window_counts_by_subject.csv"}')
display(window_summary_df.head())

X_all shape: (2000, 29, 537)
y_all counts: [1000, 1000]
subjects_arr shape: (2000,)
Wrote window summary: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-preprocess-csp-fbcsp/20260518_192347/window_counts_by_subject.csv


,subject_id,n_windows,class_counts,preprocessed_shape,resampled_samples_per_trial,crop_start_sample,crop_stop_sample,target_window_samples,effective_window_duration_s
0,1,40,"[20, 20]","(40, 29, 537)",1024,256,793,537,4.195312
1,2,40,"[20, 20]","(40, 29, 537)",1024,256,793,537,4.195312
2,3,40,"[20, 20]","(40, 29, 537)",1024,256,793,537,4.195312
3,4,40,"[20, 20]","(40, 29, 537)",1024,256,793,537,4.195312
4,5,40,"[20, 20]","(40, 29, 537)",1024,256,793,537,4.195312


## 6. CSP/FBCSP helpers

Only one CSP setup and one FBCSP setup are used here.

In [8]:
def bandpass_zero_phase(X, sfreq, l_freq, h_freq, order=5):
    sos = butter(order, [l_freq, h_freq], btype='bandpass', fs=sfreq, output='sos')
    return sosfiltfilt(sos, X, axis=-1)


def make_csp_lda(n_components=N_CSP_COMPONENTS):
    return Pipeline([
        ('csp', CSP(n_components=n_components, reg='ledoit_wolf', log=True, norm_trace=False)),
        ('lda', LinearDiscriminantAnalysis()),
    ])


def make_fbcsp_features(X_train, y_train, X_test):
    feats_train, feats_test = [], []
    for band in FBCSP_BANDS:
        Xtr = bandpass_zero_phase(X_train, SFREQ, band[0], band[1])
        Xte = bandpass_zero_phase(X_test, SFREQ, band[0], band[1])
        csp = CSP(n_components=N_CSP_COMPONENTS, reg='ledoit_wolf', log=True, norm_trace=False)
        feats_train.append(csp.fit_transform(Xtr, y_train))
        feats_test.append(csp.transform(Xte))
    return np.concatenate(feats_train, axis=1), np.concatenate(feats_test, axis=1)


def fold_metrics(y_test, pred, scores=None):
    row = {
        'accuracy': float(accuracy_score(y_test, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
        'confusion_matrix': confusion_matrix(y_test, pred, labels=[0, 1]).tolist(),
        'prediction_histogram': np.bincount(pred, minlength=2).tolist(),
    }
    if scores is not None and len(np.unique(y_test)) == 2:
        try:
            row['roc_auc'] = float(roc_auc_score(y_test, scores))
        except Exception:
            row['roc_auc'] = None
    else:
        row['roc_auc'] = None
    return row

## 7. Within-subject 5-fold CSP/FBCSP evaluation

This follows the S-JEPA-style downstream comparison logic: within-subject stratified 5-fold CV.

In [9]:
rows = []

for subj in sorted(pd.unique(subjects_arr), key=lambda x: int(x)):
    idx = np.where(subjects_arr == subj)[0]
    X = X_all[idx]
    y = y_all[idx]
    counts = np.bincount(y, minlength=2)
    log(f'Subject {subj}: X={X.shape}, class_counts={counts.tolist()}')

    if counts.min() < CV_FOLDS:
        log(f'  SKIP subject {subj}: not enough trials per class for {CV_FOLDS}-fold CV')
        continue

    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    for fold, (tr, te) in enumerate(cv.split(X, y), start=1):
        X_train_raw, X_test_raw = X[tr], X[te]
        y_train, y_test = y[tr], y[te]

        # CSP + LDA: 8-30 Hz over the S-JEPA-preprocessed/cropped windows.
        X_train = bandpass_zero_phase(X_train_raw, SFREQ, CSP_BAND[0], CSP_BAND[1])
        X_test = bandpass_zero_phase(X_test_raw, SFREQ, CSP_BAND[0], CSP_BAND[1])
        clf = make_csp_lda()
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        scores = clf.decision_function(X_test) if hasattr(clf, 'decision_function') else None
        row = {
            'subject_id': str(subj),
            'fold_id': fold,
            'model_name': 'CSP_LDA',
            'n_train': int(len(tr)),
            'n_test': int(len(te)),
            'train_class_counts': np.bincount(y_train, minlength=2).tolist(),
            'test_class_counts': np.bincount(y_test, minlength=2).tolist(),
            'feature_band': list(CSP_BAND),
            'n_csp_components': N_CSP_COMPONENTS,
        }
        row.update(fold_metrics(y_test, pred, scores))
        rows.append(row)

        # FBCSP + LDA: same family as your previous S-JEPA-style CSP baseline.
        Ftr, Fte = make_fbcsp_features(X_train_raw, y_train, X_test_raw)
        lda = LinearDiscriminantAnalysis()
        lda.fit(Ftr, y_train)
        pred = lda.predict(Fte)
        scores = lda.decision_function(Fte) if hasattr(lda, 'decision_function') else None
        row = {
            'subject_id': str(subj),
            'fold_id': fold,
            'model_name': 'FBCSP_LDA',
            'n_train': int(len(tr)),
            'n_test': int(len(te)),
            'train_class_counts': np.bincount(y_train, minlength=2).tolist(),
            'test_class_counts': np.bincount(y_test, minlength=2).tolist(),
            'n_filter_bands': len(FBCSP_BANDS),
            'n_csp_components_per_band': N_CSP_COMPONENTS,
        }
        row.update(fold_metrics(y_test, pred, scores))
        rows.append(row)

results_df = pd.DataFrame(rows)
results_path = ARTIFACT_DIR / 'cv_results.csv'
results_df.to_csv(results_path, index=False)
log(f'Wrote fold results: {results_path}')
display(results_df.head())

Subject 1: X=(40, 29, 537), class_counts=[20, 20]
Computing rank from data with rank=None
    Using tolerance 5.3 (2.2e-16 eps * 29 dim * 8.3e+14  max singular value)
    Estimated rank (data): 29
    data: rank 29 computed from 29 data channels with 0 projectors
Reducing data rank from 29 -> 29
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Computing rank from data with rank=None
    Using tolerance 8.7 (2.2e-16 eps * 29 dim * 1.3e+15  max singular value)
    Estimated rank (data): 29
    data: rank 29 computed from 29 data channels with 0 projectors
Reducing data rank from 29 -> 29
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Computing rank from data with rank=None
    Using tolerance 3.7 (2.2e-16 eps * 29 dim * 5.8e+14  max singular value)
    Estimated rank (data): 29
    data: rank 29 computed from 29 data channels with 0 projectors
Reducing data rank from 29

,subject_id,fold_id,model_name,n_train,n_test,train_class_counts,test_class_counts,feature_band,n_csp_components,accuracy,balanced_accuracy,confusion_matrix,prediction_histogram,roc_auc,n_filter_bands,n_csp_components_per_band
0,1,1,CSP_LDA,32,8,"[16, 16]","[4, 4]","[8.0, 30.0]",4.0,0.500,0.500,"[[0, 4], [0, 4]]","[0, 8]",0.0000,NaN,NaN
1,1,1,FBCSP_LDA,32,8,"[16, 16]","[4, 4]",NaN,NaN,0.125,0.125,"[[1, 3], [4, 0]]","[5, 3]",0.2500,9.0,4.0
2,1,2,CSP_LDA,32,8,"[16, 16]","[4, 4]","[8.0, 30.0]",4.0,0.625,0.625,"[[1, 3], [0, 4]]","[1, 7]",0.5000,NaN,NaN
3,1,2,FBCSP_LDA,32,8,"[16, 16]","[4, 4]",NaN,NaN,0.500,0.500,"[[2, 2], [2, 2]]","[4, 4]",0.4375,9.0,4.0
4,1,3,CSP_LDA,32,8,"[16, 16]","[4, 4]","[8.0, 30.0]",4.0,0.500,0.500,"[[1, 3], [1, 3]]","[2, 6]",0.3125,NaN,NaN


## 8. Aggregate metrics and diagnostics

In [10]:
global_metrics = (
    results_df
    .groupby('model_name')
    .agg(
        mean_accuracy=('accuracy', 'mean'),
        std_accuracy=('accuracy', 'std'),
        mean_balanced_accuracy=('balanced_accuracy', 'mean'),
        std_balanced_accuracy=('balanced_accuracy', 'std'),
        mean_roc_auc=('roc_auc', 'mean'),
        std_roc_auc=('roc_auc', 'std'),
        n_folds_total=('fold_id', 'count'),
        n_subjects=('subject_id', lambda s: int(pd.Series(s).nunique())),
    )
    .reset_index()
)

global_metrics_path = ARTIFACT_DIR / 'global_metrics.csv'
global_metrics.to_csv(global_metrics_path, index=False)
log('Global metrics:')
log(global_metrics.to_string(index=False))
display(global_metrics)

subject_metrics = (
    results_df
    .groupby(['model_name', 'subject_id'])
    .agg(
        mean_accuracy=('accuracy', 'mean'),
        std_accuracy=('accuracy', 'std'),
        mean_balanced_accuracy=('balanced_accuracy', 'mean'),
        mean_roc_auc=('roc_auc', 'mean'),
        n_folds=('fold_id', 'count'),
    )
    .reset_index()
)
subject_metrics_path = ARTIFACT_DIR / 'subject_metrics.csv'
subject_metrics.to_csv(subject_metrics_path, index=False)

collapse_rows = []
for model_name, sub in results_df.groupby('model_name'):
    n_folds = len(sub)
    n_collapsed = 0
    counts = {0: 0, 1: 0}
    for hist in sub['prediction_histogram']:
        h = hist if isinstance(hist, list) else json.loads(hist)
        nonzero = [i for i, v in enumerate(h) if v > 0]
        if len(nonzero) == 1:
            n_collapsed += 1
            counts[nonzero[0]] += 1
    collapse_rows.append({
        'model_name': model_name,
        'n_folds': int(n_folds),
        'n_collapsed_single_class': int(n_collapsed),
        'collapsed_fraction': float(n_collapsed / n_folds) if n_folds else None,
        'single_class_prediction_counts': counts,
    })
collapse_df = pd.DataFrame(collapse_rows)
collapse_path = ARTIFACT_DIR / 'collapse_diagnostics.csv'
collapse_df.to_csv(collapse_path, index=False)
log('Collapse diagnostics:')
log(collapse_df.to_string(index=False))
display(collapse_df)

Global metrics:
model_name  mean_accuracy  std_accuracy  mean_balanced_accuracy  std_balanced_accuracy  mean_roc_auc  std_roc_auc  n_folds_total  n_subjects
   CSP_LDA          0.514      0.196426                   0.514               0.196426       0.51375     0.253022            250          50
 FBCSP_LDA          0.515      0.175423                   0.515               0.175423       0.52200     0.219993            250          50


,model_name,mean_accuracy,std_accuracy,mean_balanced_accuracy,std_balanced_accuracy,mean_roc_auc,std_roc_auc,n_folds_total,n_subjects
0,CSP_LDA,0.514,0.196426,0.514,0.196426,0.51375,0.253022,250,50
1,FBCSP_LDA,0.515,0.175423,0.515,0.175423,0.52200,0.219993,250,50


Collapse diagnostics:
model_name  n_folds  n_collapsed_single_class  collapsed_fraction single_class_prediction_counts
   CSP_LDA      250                         6               0.024                   {0: 2, 1: 4}
 FBCSP_LDA      250                         8               0.032                   {0: 2, 1: 6}


,model_name,n_folds,n_collapsed_single_class,collapsed_fraction,single_class_prediction_counts
0,CSP_LDA,250,6,0.024,"{0: 2, 1: 4}"
1,FBCSP_LDA,250,8,0.032,"{0: 2, 1: 6}"


## 9. Save run metadata

In [11]:
metadata = {
    'notebook': 'liu2024_source_mat_sjepa_preprocess_csp_fbcsp',
    'source': 'original Figshare sourcedata .mat files',
    'source_extract_dir': str(SOURCE_EXTRACT_DIR),
    'n_subjects': int(subjects_df['subject_id'].nunique()),
    'source_sfreq': SOURCE_SFREQ,
    'target_sfreq': SFREQ,
    'preprocessing_order': [
        'select 29 EEG channels; drop CPz source reference, EOG, and marker',
        'concatenate trials per subject into RawArray',
        'resample to 128 Hz',
        'bandpass 0.5-40 Hz',
        'average reference',
        'reshape back to trials',
        'crop fixed 537-sample MI window starting at 2.0s',
    ],
    'target_window_samples': TARGET_WINDOW_SAMPLES,
    'effective_window_duration_s': TARGET_WINDOW_DURATION_S,
    'mi_window_start_s': MI_WINDOW_START_S,
    'mi_window_start_sample': MI_WINDOW_START_SAMPLE,
    'mi_window_stop_sample': MI_WINDOW_STOP_SAMPLE,
    'cv': {'type': 'StratifiedKFold', 'n_splits': CV_FOLDS, 'shuffle': True, 'random_state': RANDOM_STATE},
    'csp': {
        'model': 'CSP_LDA', 'feature_band': list(CSP_BAND), 'n_components': N_CSP_COMPONENTS,
        'reg': 'ledoit_wolf', 'log': True, 'norm_trace': False,
    },
    'fbcsp': {
        'model': 'FBCSP_LDA', 'bands': [list(b) for b in FBCSP_BANDS],
        'n_components_per_band': N_CSP_COMPONENTS, 'reg': 'ledoit_wolf', 'log': True, 'norm_trace': False,
    },
}

metadata_path = ARTIFACT_DIR / 'run_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
log(f'Wrote metadata: {metadata_path}')
log('Done.')
_log_handle.close()

Wrote metadata: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-preprocess-csp-fbcsp/20260518_192347/run_metadata.json
Done.
